# Extractive Summarization — TF-IDF
Metode TF-IDF: memilih kalimat dengan kata-kata paling penting (sering di artikel ini,
jarang di artikel lain). Input: `data_final_preprocessing.csv` (sudah dipreprocessing,
tidak ada preprocessing lagi di sini).

**Struktur:** Load → Stopword → Fungsi TF-IDF → ROUGE → BERTScore → Contoh → Simpan.

---
## Setup

In [27]:
!pip install rouge-score bert-score -q

In [28]:
import re
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
from nltk.tokenize import sent_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from rouge_score import rouge_scorer
from bert_score import score as bertscore
from tqdm import tqdm
tqdm.pandas()

---
## 1. Load Dataset

In [29]:
df = pd.read_csv('/kaggle/input/datasets/nazhifberlian/nlp-wikipedia-summarization/data_final_preprocessing.csv',
                 encoding='utf-8-sig')

print(f'Dataset dimuat: {len(df)} baris, {len(df.columns)} kolom')
print(f'Kolom: {list(df.columns)}')
print(f'\nDistribusi kategori:')
print(df['category'].value_counts().to_string())

Dataset dimuat: 12644 baris, 9 kolom
Kolom: ['global_id', 'id', 'title', 'category', 'article_text', 'body_word_count', 'lead_paragraph', 'lead_word_count', 'sentences']

Distribusi kategori:
category
sejarah     1976
arts        1962
artis       1947
kuliner     1947
tech        1716
biografi    1679
sains       1417


---
## 2. Stopword Removal
Stopword (kata umum seperti "yang", "di", "dan") diabaikan saat menghitung skor TF-IDF
agar fokus ke kata bermakna. Diterapkan di dalam `TfidfVectorizer`.

In [30]:
STOPWORDS_ID = set("""
yang di ke dari dan atau pada untuk dengan adalah ini itu dalam tidak akan
sebagai juga oleh karena sudah dapat telah agar bisa saat para suatu setelah
hingga maupun sebuah merupakan namun yaitu serta secara hanya masih lebih tersebut
""".split())

print(f'Jumlah stopword: {len(STOPWORDS_ID)}')

Jumlah stopword: 40


---
## 3. Fungsi Summarization TF-IDF

In [31]:
def split_sentences(text):
    sentences = sent_tokenize(str(text))
    return [s.strip() for s in sentences if len(s.split()) >= 4]

def adaptive_n(text):
    n = len(str(text).split())
    if n < 500:  return 2
    if n < 1500: return 3
    return 4

def summarize_tfidf(text, n_sentences=3):
    sentences = split_sentences(text)
    if len(sentences) <= n_sentences:
        return ' '.join(sentences)
    vectorizer = TfidfVectorizer(stop_words=list(STOPWORDS_ID), lowercase=True)
    tfidf_matrix = vectorizer.fit_transform(sentences)
    scores = tfidf_matrix.sum(axis=1).A1
    top_idx = sorted(scores.argsort()[-n_sentences:])
    return ' '.join(sentences[i] for i in top_idx)

print('Fungsi TF-IDF siap.')

Fungsi TF-IDF siap.


### Generate ringkasan untuk seluruh dataset

In [32]:
print('Generating ringkasan TF-IDF...')
start = time.time()

df['summary_tf_idf'] = df['article_text'].progress_apply(
    lambda x: summarize_tfidf(x, n_sentences=adaptive_n(x)))

time_tfidf = time.time() - start
print(f'Selesai dalam {time_tfidf:.1f} detik ({time_tfidf/len(df):.3f} detik/artikel)')

Generating ringkasan TF-IDF...


100%|██████████| 12644/12644 [00:33<00:00, 377.32it/s]

Selesai dalam 33.5 detik (0.003 detik/artikel)


---
## 4. Evaluasi ROUGE

In [33]:
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=False)

r1, r2, rl = [], [], []
for pred, ref in zip(df['summary_tf_idf'], df['lead_paragraph']):
    s = scorer.score(str(ref), str(pred))
    r1.append(s['rouge1'].fmeasure)
    r2.append(s['rouge2'].fmeasure)
    rl.append(s['rougeL'].fmeasure)

rouge_tfidf = (np.mean(r1), np.mean(r2), np.mean(rl))

print('Evaluasi ROUGE — TF-IDF')
print(f'{"Metrik":<10} | {"Skor":>8}')
print('-' * 22)
print(f'{"ROUGE-1":<10} | {rouge_tfidf[0]:>8.4f}')
print(f'{"ROUGE-2":<10} | {rouge_tfidf[1]:>8.4f}')
print(f'{"ROUGE-L":<10} | {rouge_tfidf[2]:>8.4f}')

Evaluasi ROUGE — TF-IDF
Metrik     |     Skor
----------------------
ROUGE-1    |   0.2088
ROUGE-2    |   0.0472
ROUGE-L    |   0.1211


### ROUGE-L per Kategori

In [34]:
rows = []
for kat in sorted(df['category'].unique()):
    sub = df[df['category'] == kat]
    scores = [scorer.score(str(ref), str(pred))['rougeL'].fmeasure
              for pred, ref in zip(sub['summary_tf_idf'], sub['lead_paragraph'])]
    rows.append({'kategori': kat, 'jumlah': len(sub), 'rougeL': np.mean(scores)})

hasil_kategori = pd.DataFrame(rows).sort_values('rougeL', ascending=False)
print('ROUGE-L per kategori (TF-IDF):')
print(hasil_kategori.round(4).to_string(index=False))

ROUGE-L per kategori (TF-IDF):
kategori  jumlah  rougeL
 sejarah    1976  0.1285
    arts    1962  0.1239
 kuliner    1947  0.1232
    tech    1716  0.1222
   artis    1947  0.1176
   sains    1417  0.1171
biografi    1679  0.1128


---
## 5. Evaluasi BERTScore

In [35]:
# BERTScore mengukur kemiripan makna (bukan overlap kata). Berat, jadi pakai sampel.
# lang='id' otomatis memilih model multilingual (berbeda dari metode extractive BERT).
SAMPLE_SIZE_BS = 500
sample_bs = df.sample(n=min(SAMPLE_SIZE_BS, len(df)), random_state=42).reset_index(drop=True)

cands = sample_bs['summary_tf_idf'].astype(str).tolist()
refs  = sample_bs['lead_paragraph'].astype(str).tolist()

P, R, F1 = bertscore(cands, refs, lang='id', verbose=False)
bertscore_tfidf = (P.mean().item(), R.mean().item(), F1.mean().item())

print(f'Evaluasi BERTScore — TF-IDF (sampel N={len(sample_bs)})')
print(f'{"Metrik":<12} | {"Skor":>8}')
print('-' * 24)
print(f'{"Precision":<12} | {bertscore_tfidf[0]:>8.4f}')
print(f'{"Recall":<12} | {bertscore_tfidf[1]:>8.4f}')
print(f'{"F1":<12} | {bertscore_tfidf[2]:>8.4f}')

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Evaluasi BERTScore — TF-IDF (sampel N=500)
Metrik       |     Skor
------------------------
Precision    |   0.6632
Recall       |   0.6692
F1           |   0.6656


---
## 6. Contoh Ringkasan Hasil TF-IDF

In [43]:
N_CONTOH = 3
sample_idx = df.sample(N_CONTOH, random_state=42).index

for i, idx in enumerate(sample_idx):
    row = df.loc[idx]
    print('=' * 70)
    print(f'Artikel {i+1}: [{row["category"]}] {row["title"]}')
    print('-' * 20)
    print(f'Ground Truth (lead_paragraph):')
    print(f'{str(row["lead_paragraph"])[:200]}...')
    print('-' * 20)
    print(f'Ringkasan TF-IDF:')
    print(f'{str(row["summary_tf_idf"])[:200]}')
    print()

Artikel 1: [sains] Atom Bakhidrogen
--------------------
Ground Truth (lead_paragraph):
suatu ion lirhidrogen bahasa inggris hydrogenlike ioncode en is deprecated  disebut pula ion mirip hidrogen merupakan inti atom yang memiliki satu elektron dan karenanya isoelektronik dengan hidrogen....
--------------------
Ringkasan TF-IDF:
suatu orbital atom bakhidrogen secara unik diidentifikasi melalui nilai bilangan kuantum utama n, bilangan kuantum momentum sudut l, serta bilangan kuantum magnetik m. eigennilai energinya tidak berga

Artikel 2: [kuliner] Jeruk Bergamot
--------------------
Ground Truth (lead_paragraph):
jeruk bergamot adalah buah jeruk wangi dengan warna kuning atau hijau mirip dengan jeruk nipis, tergantung pada kematangannya. penelitian genetika tentang asal muasal leluhur kultivar jeruk yang masih...
--------------------
Ringkasan TF-IDF:
produksinya dilakukan dalam skala besar di wilayah pesisir laut ionia di provinsi reggio di calabria, italia, sedemikian rupa sehingga m

---
## 7. Simpan summary_tf_idf

In [37]:
output_cols = ['global_id', 'title', 'category', 'summary_tf_idf',
               'lead_paragraph', 'body_word_count']
hasil = df[[c for c in output_cols if c in df.columns]]

hasil.to_csv('hasil_summary_tfidf.csv', index=False, encoding='utf-8-sig')
print(f'Tersimpan: hasil_summary_tfidf.csv ({len(hasil)} baris)')
print(f'Kolom: {list(hasil.columns)}')

Tersimpan: hasil_summary_tfidf.csv (12644 baris)
Kolom: ['global_id', 'title', 'category', 'summary_tf_idf', 'lead_paragraph', 'body_word_count']
